In [1]:
!pip install autogluon.tabular[0] autogluon[tabarena]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import TargetEncoder
from autogluon.tabular import TabularDataset, TabularPredictor
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import warnings
warnings.filterwarnings('ignore')

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e2/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e2/train.csv
/kaggle/input/competitions/playground-series-s6e2/test.csv


In [3]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/train.csv')
train['source'] = 'train'
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/sample_submission.csv')

NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if train[col].nunique() > 40]

for df in [train, test]:
    int_cols = df.select_dtypes(include=['int64']).columns.to_list()
    float_cols = df.select_dtypes(include=['float64']).columns.to_list()
    df[int_cols] = df[int_cols].astype('int32')
    df[float_cols] = df[float_cols].astype('float32')

FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source', 'index', 'strat_feature']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
# X = X.fillna(0)
# X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
# y_org = org[CONFIG.TARGET]

X_test = test[FEATURES]
# X_test = X_test.fillna(0)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
13


In [4]:
BASE_FEATURES = ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

for c in BASE_FEATURES:
    n = f'{c}_mean_te'
    TE = TargetEncoder(cv=5, random_state=42, shuffle=True)
    X[n] = TE.fit_transform(pd.DataFrame(X[c]), y).flatten()
    X_test[n] = TE.transform(pd.DataFrame(X_test[c])).flatten()

CATS = []
for col in NUMS:
    n = f'{col}_cat'
    for df in [X, X_test]:
        df[n] = df[col].astype(str).astype('category')
    CATS.append(n)

In [5]:
tabular_df = TabularDataset(pd.concat([X, y], axis=1))
label = CONFIG.TARGET
aml = TabularPredictor(label=label, eval_metric='roc_auc', problem_type='binary', path="AutogluonModelsExtremePreset", learner_kwargs={'random_state': 42})

In [6]:
aml.fit(train_data=tabular_df,
    time_limit=3600*11,
    presets='extreme',
    num_bag_folds=5,         
    num_bag_sets=1,             
    # num_stack_levels=7,   
    refit_full=False,
    # set_best_to_refit_full=True, 
    ag_args_fit={
                'num_gpus': 1,
                'num_cpus': 4}, 
    # Keep random seed consistency
    # feature_generator_kwargs={'fixed_random_state': 42}
       )

Preset alias specified: 'extreme' maps to 'extreme_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.0+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 15.89/15.89 GB
Total GPU Memory:   Free: 15.89 GB, Allocated: 0.00 GB, Total: 15.89 GB
GPU Count:          1
Memory Avail:       29.45 GB / 31.35 GB (93.9%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['extreme']
Using hyperparameters preset: hyperparameters='zeroshot_2025_12_18_gpu'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=5, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 39600s
AutoGluon will save models to "/kaggle/working/AutogluonModelsExtremePreset"
Train Data Rows:    630000
Train Data Columns: 39
Label C

In [7]:
leaderboard = aml.leaderboard()
leaderboard.to_csv('leaderboard.csv', index=False)

In [8]:
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.955730,roc_auc,99.015512,12135.561227,0.121980,35.545372,2,True,13
1,TabM_r124_BAG_L1,0.955689,roc_auc,6.250654,2038.085461,6.250654,2038.085461,1,True,3
2,CatBoost_c1_BAG_L1,0.955670,roc_auc,2.352790,200.457567,2.352790,200.457567,1,True,2
3,TabM_r184_BAG_L1,0.955629,roc_auc,18.968348,4212.587250,18.968348,4212.587250,1,True,8
4,TabM_r69_BAG_L1,0.955618,roc_auc,19.328380,4627.382325,19.328380,4627.382325,1,True,6
5,TabM_r34_BAG_L1,0.955593,roc_auc,22.693558,5173.267623,22.693558,5173.267623,1,True,10
6,LightGBM_r37_BAG_L1,0.955545,roc_auc,48.628181,475.617955,48.628181,475.617955,1,True,7
7,TabM_r99_BAG_L1,0.955523,roc_auc,28.800602,6775.044625,28.800602,6775.044625,1,True,1
8,LightGBMPrep_r21_BAG_L1,0.955515,roc_auc,29.150629,281.797537,29.150629,281.797537,1,True,5
9,LightGBM_r73_BAG_L1,0.955391,roc_auc,48.265591,448.750231,48.265591,448.750231,1,True,4


In [9]:
model_names = aml.model_names(can_infer=True)
oof_dict = {}
test_dict = {}
for model_name in model_names:
    if not model_name.endswith('_FULL'):
        try:
            oof_probs = aml.predict_proba_oof(model=model_name).iloc[:, 1]
            oof_dict[model_name] = oof_probs.values

            test_probs = aml.predict_proba(X_test, model=model_name).iloc[:, 1]
            test_dict[model_name] = test_probs.values

            print(f'{model_name} done ...')

        except Exception as e:
            print(f'Could not extract {model_name}: {e}')

TabM_r99_BAG_L1 done ...
CatBoost_c1_BAG_L1 done ...
TabM_r124_BAG_L1 done ...
LightGBM_r73_BAG_L1 done ...
LightGBMPrep_r21_BAG_L1 done ...
TabM_r69_BAG_L1 done ...
LightGBM_r37_BAG_L1 done ...
TabM_r184_BAG_L1 done ...
LightGBM_r162_BAG_L1 done ...
TabM_r34_BAG_L1 done ...
LightGBM_r57_BAG_L1 done ...
LightGBM_r33_BAG_L1 done ...
WeightedEnsemble_L2 done ...


In [10]:
all_model_oofs = pd.DataFrame(oof_dict)
all_model_test = pd.DataFrame(test_dict)

all_model_oofs.to_csv('autogluon_all_oofs.csv', index=False)
all_model_test.to_csv('autogluon_all_test.csv', index=False)

print("\n--- Process Complete ---")
print(f"OOF Shape: {all_model_oofs.shape}")
print(f"Test Shape: {all_model_test.shape}")


--- Process Complete ---
OOF Shape: (630000, 13)
Test Shape: (270000, 13)
